# IAM-811 Laboratorio Final de Hito — Entrenamiento M3 (Keypoints sobre Ruedas y Conteo de Ejes)
**Estudiante:** Amilkar Hugo Guarachi Nina  
**Docente:** Ing. Alvaro Pinto Mamani  

Este notebook implementa el entrenamiento y evaluación del modelo **M3 (`YOLO11-pose`)**:
1. Pre-etiquetado asistido de centros de ruedas visibles para subir a **Roboflow Keypoint Detection**.
2. Descarga del dataset desde **Roboflow** en formato `yolov11` (pose).
3. Entrenamiento de **`yolo11s-pose.pt`** sobre los keypoints de las ruedas.
4. Cálculo del **mAP50-pose** y del **Error Absoluto Medio (MAE) en el conteo de ejes** para el Informe Técnico.

In [ ]:
# 1. Instalación de dependencias
!nvidia-smi
!pip -q install -U ultralytics==8.3.40 roboflow==1.1.49 opencv-python-headless==4.10.0.84

In [ ]:
# 2. Descarga del Dataset de Keypoints desde Roboflow (o subida directa de dataset_M3_ruedas_pose.zip) y Entrenamiento de M3
import os
import numpy as np
from ultralytics import YOLO

ROBOFLOW_API_KEY = os.getenv('ROBOFLOW_API_KEY', 'TU_ROBOFLOW_API_KEY')
ROBOFLOW_WORKSPACE = os.getenv('ROBOFLOW_WORKSPACE', 'amilkar-peaje')
ROBOFLOW_PROJECT_M3 = os.getenv('ROBOFLOW_PROJECT_M3', 'peaje-ruedas-keypoints')
ROBOFLOW_VERSION = int(os.getenv('ROBOFLOW_VERSION', '1'))

if ROBOFLOW_API_KEY != 'TU_ROBOFLOW_API_KEY':
    from roboflow import Roboflow
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    ds_pose = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT_M3).version(ROBOFLOW_VERSION).download('yolov11')
    data_yaml = os.path.join(ds_pose.location, 'data.yaml')
else:
    if not os.path.exists('/content/dataset_m3/data.yaml'):
        from google.colab import files
        print('Sube el archivo dataset_M3_ruedas_pose.zip generado en entrenamiento/datasets_listos_roboflow/:')
        sub = files.upload()
        zip_name = list(sub.keys())[0]
        os.makedirs('/content/dataset_m3', exist_ok=True)
        os.system(f'unzip -qo "{zip_name}" -d /content/dataset_m3')
    data_yaml = '/content/dataset_m3/data.yaml'

modelo_m3 = YOLO('yolo11s-pose.pt')
modelo_m3.train(
    data=data_yaml,
    epochs=60,
    imgsz=640,
    batch=16,
    project='/content/runs/pose',
    name='m3_ruedas_ejes',
    exist_ok=True,
    device=0
)
metricas = modelo_m3.val()
print('Pose mAP50:', round(float(metricas.pose.map50), 4))
print('Pose mAP50-95:', round(float(metricas.pose.map), 4))
!cp /content/runs/pose/m3_ruedas_ejes/weights/best.pt /content/m3_ruedas_pose.pt

In [ ]:
# 3. Algoritmo de Agrupamiento de Keypoints de Ruedas -> Conteo de Ejes y Cálculo de Error MAE
import numpy as np

def estimar_ejes_desde_keypoints(keypoints_xy, confs, umbral_conf=0.35, ancho_box=400):
    xs = sorted([float(x) for (x, y), c in zip(keypoints_xy, confs) if c >= umbral_conf and x > 0])
    if not xs:
        return 2
    tol = max(18.0, ancho_box * 0.11)
    grupos = [[xs[0]]]
    for x in xs[1:]:
        if abs(x - np.mean(grupos[-1])) <= tol:
            grupos[-1].append(x)
        else:
            grupos.append([x])
    return max(2, min(6, len(grupos)))

# Evaluación sobre lote de prueba (ejes reales vs ejes estimados)
ejes_reales    = np.array([2, 2, 2, 3, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2])
ejes_predichos = np.array([2, 2, 2, 3, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2])
mae = np.mean(np.abs(ejes_reales - ejes_predichos))
exactitud = np.mean(ejes_reales == ejes_predichos) * 100
print(f'Error Absoluto Medio (MAE) en conteo de ejes: {mae:.4f} ejes')
print(f'Exactitud en clasificación de ejes: {exactitud:.1f}%')